# 01 - Exploracion de datos

Aca quiero entender bien con que estoy trabajando antes de meterme a entrenar cualquier cosa.
Voy a revisar cuantas imagenes hay por clase, como se ven, y que tan desbalanceado esta el dataset.
Tambien defino el split que voy a usar en todos los notebooks siguientes.

In [ ]:
!pip install tensorflow-datasets --quiet

In [ ]:
import os

# carpeta local donde guardo los artefactos generados en esta sesion
WORK_PATH = '/content/plantvillage'
os.makedirs(WORK_PATH, exist_ok=True)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import json
from collections import Counter

print('tensorflow:', tf.__version__)
print('gpu disponible:', tf.config.list_physical_devices('GPU'))

tf.random.set_seed(42)
np.random.seed(42)

## Carga del dataset

Uso tensorflow_datasets porque se descarga solo y no necesito cuenta de Kaggle.
Defino el split aca: 70% para entrenar, 15% para validar durante el entrenamiento,
y 15% que no toco hasta el final para evaluar.

In [ ]:
(ds_train, ds_val, ds_test), info = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
    read_config=tfds.ReadConfig(shuffle_seed=42)
)

NUM_CLASSES = info.features['label'].num_classes
CLASS_NAMES = info.features['label'].names
TOTAL       = info.splits['train'].num_examples

print('total de imagenes:', TOTAL)
print('numero de clases:', NUM_CLASSES)
print('ejemplo de clases:', CLASS_NAMES[:4], '...')

## Distribucion de clases

Quiero ver cuantas imagenes tiene cada clase porque esto afecta directamente
como voy a entrenar. Si hay clases con muy pocas imagenes el modelo va a
tener problemas aprendiendolas.

In [ ]:
# cuento las imagenes por clase en el split de train
conteo = Counter()
for _, label in tfds.as_numpy(ds_train):
    conteo[CLASS_NAMES[label]] += 1

clases_ordenadas = sorted(conteo.items(), key=lambda x: x[1], reverse=True)
nombres, cantidades = zip(*clases_ordenadas)

print('clase con mas imagenes: ', nombres[0], '->', cantidades[0])
print('clase con menos imagenes:', nombres[-1], '->', cantidades[-1])
print('diferencia entre ambas: ', round(cantidades[0] / cantidades[-1], 1), 'x')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 13))

ax.barh(range(len(nombres)), cantidades, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(nombres)))
ax.set_yticklabels(nombres, fontsize=8)
ax.set_xlabel('imagenes en train (70%)')
ax.set_title('cuantas imagenes hay por clase')
ax.axvline(np.mean(cantidades), color='red', linestyle='--', alpha=0.6,
           label=f'promedio: {int(np.mean(cantidades))}')
ax.legend()

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/distribucion_clases.png', dpi=130)
plt.show()

## Muestras del dataset

Quiero ver como se ven las imagenes antes de procesarlas.
Tomo una imagen de distintas clases para tener una idea variada.

In [ ]:
# indices que quiero mostrar, distribuidos por todo el rango de clases
indices_muestra = [0, 5, 10, 15, 20, 25, 30, 35, 37]

muestras = {}
for imagen, label in tfds.as_numpy(ds_train.take(3000)):
    idx = int(label)
    if idx in indices_muestra and idx not in muestras:
        muestras[idx] = imagen
    if len(muestras) == len(indices_muestra):
        break

fig, ejes = plt.subplots(3, 3, figsize=(11, 11))

for ax, idx in zip(ejes.flatten(), indices_muestra):
    if idx in muestras:
        ax.imshow(muestras[idx])
        # el nombre viene con guiones bajos, los reemplazo para que se lea mejor
        nombre = CLASS_NAMES[idx].replace('___', ' - ').replace('_', ' ')
        ax.set_title(nombre, fontsize=7)
    ax.axis('off')

plt.suptitle('muestra de imagenes del dataset', fontsize=12)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/muestras.png', dpi=130)
plt.show()

## Estadisticas basicas

Calculo la media y desviacion por canal RGB sobre una muestra.
Esto me va a servir para saber si necesito normalizar y como.

In [ ]:
MUESTRA = 800
pixeles = []

for i, (img, _) in enumerate(tfds.as_numpy(ds_train)):
    if i >= MUESTRA:
        break
    pixeles.append(img.astype(np.float32) / 255.0)

arr = np.array(pixeles)
media = arr.mean(axis=(0, 1, 2))
std   = arr.std(axis=(0, 1, 2))

print('resolucion nativa: 256x256 px')
print('media por canal [R G B]:', media.round(3))
print('std   por canal [R G B]:', std.round(3))

## Guardar info para los otros notebooks

Guardo los nombres de clases, conteos y estadisticas en un json para reusar en los otros notebooks.

In [ ]:
info_dataset = {
    'num_classes' : NUM_CLASSES,
    'class_names' : CLASS_NAMES,
    'total_images': TOTAL,
    'media_rgb'   : media.tolist(),
    'std_rgb'     : std.tolist(),
    'conteo_train': dict(conteo),
    'split'       : '70-15-15'
}

ruta_json = f'{WORK_PATH}/dataset_info.json'
with open(ruta_json, 'w') as f:
    json.dump(info_dataset, f, indent=2)

print('archivo guardado en:', ruta_json)
print()
print('split usado: 70% train / 15% val / 15% test')